# Exploratory Data Analysis: Week Approach 🇬🇧 [🇮🇹](02_eda_week_approach_IT.ipynb)

## Runner Injury Risk Prediction — Lovdal et al. (2021) Replication

This notebook explores the **week-level training load dataset**, the second temporal
approach used by Lovdal et al. to predict injury risk in competitive runners.

The week approach trades **temporal granularity** for **richer feature aggregation**:
instead of 7 daily snapshots of 10 raw features, we get **3 weekly summaries** of
22 aggregated features (min, max, avg of subjective metrics; volume totals; session
counts) plus **3 relative km ratios** that capture week-over-week training load changes.

**Critical difference**: The target variable is **continuous** (0.0 to ~1.5+),
representing an injury score that must be **binarized** at a chosen threshold for
classification. This threshold choice directly affects class balance and model performance.

### Key questions

1. What does the **continuous target distribution** look like?
2. How does the **binarization threshold** affect class balance?
3. What do the **relative km ratios** reveal about training load changes?
4. How does this approach compare to the **day approach** (Notebook 01)?

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Ensure src/ is importable when running from notebooks/
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    ATHLETE_ID_COL,
    DATE_COL,
    INJURY_COL,
    N_WEEK_BLOCKS,
    RANDOM_SEED,
    WEEK_FEATURES,
    WEEK_INJURY_THRESHOLD,
    WEEK_RELATIVE_FEATURES,
)
from src.data_loading import get_feature_columns, load_day_data, load_week_data
from src.preprocessing.week_preprocessor import binarize_target
from src.utils.logging_config import setup_logging
from src.utils.plotting import INJURY_PALETTE, PALETTE, save_figure, set_style
from src.utils.reproducibility import set_global_seed

# --- Reproducibility & style ---
set_global_seed(RANDOM_SEED)
setup_logging()
set_style()

## 1. Data Loading and Shape Inspection

In [ ]:
df_week = load_week_data()
feature_cols_week = get_feature_columns(df_week)

print(f"Shape: {df_week.shape[0]:,} rows × {df_week.shape[1]} columns")
print(f"Feature columns: {len(feature_cols_week)}")
print(f"  - Block features: {N_WEEK_BLOCKS} blocks × {len(WEEK_FEATURES)} features = {N_WEEK_BLOCKS * len(WEEK_FEATURES)}")
print(f"  - Relative ratios: {len(WEEK_RELATIVE_FEATURES)}")
print(f"Metadata columns: {ATHLETE_ID_COL}, {INJURY_COL}, {DATE_COL}")
print(f"\nUnique athletes: {df_week[ATHLETE_ID_COL].nunique()}")
print(f"Memory usage: {df_week.memory_usage(deep=True).sum() / 1e6:.1f} MB")

In [ ]:
df_week.head()

### Data structure note

The 22 weekly features capture richer information than the 10 daily features:

- **Counts**: `nr_sessions`, `nr_rest_days`, `nr_tough_sessions`, `nr_days_with_interval_session`, `nr_strength_trainings`
- **Volume totals**: `total_kms`, `total_km_z3_4`, `total_km_z5_t1_t2`, `total_km_z3_z4_z5_t1_t2`, `total_hours_alternative_training`
- **Maxima**: `max_km_one_day`, `max_km_z3_4_one_day`, `max_km_z5_t1_t2_one_day`
- **Subjective summaries**: `avg/min/max` of `exertion`, `training_success`, `recovery`

The **min/max/avg triples** are particularly interesting — they capture **within-week
variation** that the day approach misses. An athlete with `avg_exertion = 5` and
`max_exertion = 9` had a very different week than one with `avg_exertion = 5` and
`max_exertion = 6`.

---

## 2. Continuous Target Distribution

Unlike the day approach (binary 0/1), the week approach has a **continuous injury
score**. Understanding this distribution is essential before choosing a binarization
threshold.

In [ ]:
injury_scores = df_week[INJURY_COL]

print("Continuous injury score statistics:\n")
print(f"  Mean:       {injury_scores.mean():.4f}")
print(f"  Median:     {injury_scores.median():.4f}")
print(f"  Std:        {injury_scores.std():.4f}")
print(f"  Min:        {injury_scores.min():.4f}")
print(f"  Max:        {injury_scores.max():.4f}")
print("\nPercentiles:")
for p in [25, 50, 75, 90, 95, 99]:
    print(f"  {p}th: {injury_scores.quantile(p / 100):.4f}")
print(f"\nExactly 0.0: {(injury_scores == 0).sum():,} ({(injury_scores == 0).mean() * 100:.2f}%)")
print(f"Above 0.5:   {(injury_scores >= 0.5).sum():,} ({(injury_scores >= 0.5).mean() * 100:.2f}%)")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.hist(injury_scores, bins=100, color=PALETTE["primary"], alpha=0.7, edgecolor="white")

# Threshold lines
thresholds = {0.1: ("dotted", PALETTE["neutral"]), 0.25: ("dashed", PALETTE["secondary"]), 0.5: ("solid", PALETTE["negative"])}
for t, (ls, color) in thresholds.items():
    ax.axvline(t, linestyle=ls, color=color, linewidth=2, label=f"Threshold = {t}")

ax.set_xlabel("Injury score (continuous)")
ax.set_ylabel("Count")
ax.set_title("Continuous Target Distribution — Week Approach", fontweight="bold")
ax.legend()
sns.despine()

save_figure(fig, "02_continuous_target_distribution", subdir="eda", close=False)
plt.show()
plt.close(fig)

### Interpretation

The continuous target distribution is **heavily right-skewed** with a massive spike
at 0.0 — the vast majority of weeks show no injury signal.

The continuous nature provides **more information** than the binary day target:
instead of a hard yes/no, we see gradations of injury risk. Values near 0 suggest
minimal risk, while values above 0.5 indicate meaningful injury involvement.

**The threshold choice matters**: Where we draw the binary line determines the
class balance and, by extension, what the model learns to detect. The next section
quantifies this trade-off.

---

## 3. Threshold Sensitivity: How Binarization Changes the Problem

The paper uses **0.5** as the binarization threshold (ADR-002). But this choice is
not obvious — lower thresholds capture milder injury signals but increase the positive
class size. Higher thresholds focus on more severe injuries but reduce the number
of positive samples.

This analysis documents how class balance shifts across thresholds.

In [ ]:
thresholds = [0.1, 0.25, 0.5, 0.75, 1.0]

sensitivity_data: list[dict[str, float]] = []
for t in thresholds:
    y_bin = binarize_target(injury_scores, threshold=t)
    n_pos = int(y_bin.sum())
    n_total = len(y_bin)
    pct_pos = n_pos / n_total * 100
    ratio = (n_total - n_pos) / n_pos if n_pos > 0 else float("inf")
    sensitivity_data.append({
        "threshold": t,
        "n_positive": n_pos,
        "pct_positive": pct_pos,
        "neg_pos_ratio": ratio,
    })

sensitivity_df = pd.DataFrame(sensitivity_data)
print("Threshold sensitivity analysis:\n")
print(sensitivity_df.to_string(index=False, float_format="{:.2f}".format))

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5))

color1 = PALETTE["primary"]
ax1.plot(
    sensitivity_df["threshold"],
    sensitivity_df["pct_positive"],
    color=color1,
    marker="o",
    linewidth=2,
    markersize=8,
    label="Positive class %",
)
ax1.set_xlabel("Binarization threshold")
ax1.set_ylabel("Positive class (%)", color=color1)
ax1.tick_params(axis="y", labelcolor=color1)

# Annotate each point
for _, row in sensitivity_df.iterrows():
    ax1.annotate(
        f"{row['pct_positive']:.1f}%",
        (row["threshold"], row["pct_positive"]),
        textcoords="offset points",
        xytext=(0, 12),
        ha="center",
        fontsize=9,
        fontweight="bold",
        color=color1,
    )

ax2 = ax1.twinx()
color2 = PALETTE["negative"]
ax2.plot(
    sensitivity_df["threshold"],
    sensitivity_df["neg_pos_ratio"],
    color=color2,
    marker="s",
    linewidth=2,
    markersize=8,
    linestyle="--",
    label="Neg:Pos ratio",
)
ax2.set_ylabel("Negative:Positive ratio", color=color2)
ax2.tick_params(axis="y", labelcolor=color2)

# Highlight the paper's threshold
ax1.axvline(0.5, color=PALETTE["neutral"], linestyle=":", linewidth=1.5, alpha=0.7)
ax1.text(0.52, ax1.get_ylim()[1] * 0.9, "Paper's\nthreshold", fontsize=9, color=PALETTE["neutral"])

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="center right")

ax1.set_title("Threshold Sensitivity — Effect on Class Balance", fontweight="bold")
fig.tight_layout()

save_figure(fig, "02_threshold_sensitivity", subdir="eda", close=False)
plt.show()
plt.close(fig)

### Interpretation

The threshold sensitivity analysis reveals the **fundamental trade-off** in binarizing
a continuous target:

- **At threshold 0.1**: More observations are labeled as "injured," creating a less
  extreme imbalance. But many of these may be **mild signals** — minor discomfort or
  precautionary reports, not actual injuries.

- **At threshold 0.5 (paper's choice)**: A more conservative definition of injury.
  The imbalance is less extreme than the day approach's 1.36%, making the problem
  slightly easier to model.

- **At threshold 1.0**: Only the most severe injuries are captured, potentially too
  few for the model to learn from.

**Why 0.5?** The paper chose this threshold as a balance between sensitivity (catching
real injuries) and specificity (not labeling minor issues as injuries). We follow their
convention to enable direct comparison of our results (ADR-002).

---

## 4. Relative Km Ratios: Week-over-Week Load Changes

The three ratio features are the **week approach's unique asset**:
- `rel_total_kms_week_0_1`: Week 0 km / Week 1 km
- `rel_total_kms_week_0_2`: Week 0 km / Week 2 km
- `rel_total_kms_week_1_2`: Week 1 km / Week 2 km

A ratio **> 1.0** means the numerator week had higher volume (training spike).
A ratio **< 1.0** means volume decreased (deloading).

These features directly encode the **acute:chronic workload ratio (ACWR)** concept
from sports science — one of the strongest predictors of injury risk.

In [ ]:
# Binarize target at paper threshold for this analysis
df_week["injury_binary"] = binarize_target(df_week[INJURY_COL], threshold=WEEK_INJURY_THRESHOLD)

print("Ratio feature statistics:\n")
for feat in WEEK_RELATIVE_FEATURES:
    series = df_week[feat]
    vals = series.dropna()
    zero_or_nan_pct = ((series == 0) | series.isna()).mean() * 100
    print(f"{feat}:")
    print(f"  Mean: {vals.mean():.3f}, Median: {vals.median():.3f}, Std: {vals.std():.3f}")
    print(f"  Min: {vals.min():.3f}, Max: {vals.max():.3f}")
    print(f"  % above 1.0 (load increase): {(vals > 1.0).mean() * 100:.1f}%")
    print(f"  % exactly 0.0 or NaN: {zero_or_nan_pct:.1f}%")
    print()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

short_names = ["Week 0/1", "Week 0/2", "Week 1/2"]

for ax, feat, name in zip(axes, WEEK_RELATIVE_FEATURES, short_names):
    for label, color, lbl in [
        (0, INJURY_PALETTE[0], "No Injury"),
        (1, INJURY_PALETTE[1], "Injury"),
    ]:
        subset = df_week[df_week["injury_binary"] == label][feat].dropna()
        ax.hist(
            subset,
            bins=50,
            alpha=0.5,
            color=color,
            label=f"{lbl} (n={len(subset):,})",
            density=True,
            edgecolor="white",
        )

    ax.axvline(1.0, color="black", linestyle="--", linewidth=1.2, label="Ratio = 1.0 (equal load)")
    ax.set_xlabel("Km ratio")
    ax.set_ylabel("Density")
    ax.set_title(f"Relative Km Ratio — {name}", fontweight="bold")
    ax.legend(fontsize=8)
    sns.despine(ax=ax)

fig.suptitle("Week-over-Week Km Ratio Distributions by Injury Status", fontweight="bold", fontsize=14, y=1.03)
fig.tight_layout()

save_figure(fig, "02_ratio_distributions", subdir="eda", close=False)
plt.show()
plt.close(fig)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, feat, name in zip(axes, WEEK_RELATIVE_FEATURES, short_names):
    # Use hexbin for dense scatter
    valid = df_week[[feat, INJURY_COL]].dropna()
    hb = ax.hexbin(
        valid[feat],
        valid[INJURY_COL],
        gridsize=40,
        cmap="Blues",
        mincnt=1,
    )

    ax.axhline(WEEK_INJURY_THRESHOLD, color=PALETTE["negative"], linestyle="--", linewidth=1.5, label=f"Threshold = {WEEK_INJURY_THRESHOLD}")
    ax.axvline(1.0, color=PALETTE["neutral"], linestyle=":", linewidth=1.2, alpha=0.7)
    ax.set_xlabel(f"Km ratio — {name}")
    ax.set_ylabel("Injury score (continuous)")
    ax.set_title(f"Ratio vs Injury — {name}", fontweight="bold")
    ax.legend(fontsize=8)

fig.suptitle("Km Ratios vs Continuous Injury Score", fontweight="bold", fontsize=14, y=1.03)
fig.tight_layout()

save_figure(fig, "02_ratio_vs_injury", subdir="eda", close=False)
plt.show()
plt.close(fig)

### Interpretation: The Acute:Chronic Workload Ratio

The ratio features encode the **ACWR concept** from sports science:

- **Ratio > 1.0 = training spike**: The athlete increased training volume compared to
  the reference week. This is the "danger zone" identified by Blanch and Gabbett (2016):
  rapid load increases (ACWR > 1.5) are associated with **2–4× elevated injury risk**.

- **Ratio < 1.0 = deloading**: The athlete reduced volume. While generally protective,
  very low ratios (< 0.5) can indicate detraining, which also increases injury risk
  when normal training resumes.

- **Ratio ≈ 1.0 = stable load**: Consistent week-to-week training — the safest zone.

**What to look for in the histograms:**
- If the **injured group** shows a wider distribution (more extreme ratios on both sides),
  it confirms that load variability — both spikes and drops — is a risk factor.
- If the injured group is shifted right (> 1.0), it points specifically to load spikes.

This is the **domain expertise differentiator** of this analysis: we're not just
running statistics on features — we're interpreting them through the lens of
training load management theory.

---

## 5. Week Feature Distributions (Week 0)

In [ ]:
week0_cols = [f"week_0_{feat}" for feat in WEEK_FEATURES]

n_feats = len(WEEK_FEATURES)
n_cols = 4
n_rows = (n_feats + n_cols - 1) // n_cols  # ceiling division

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 3.5))
axes = axes.flatten()

for i, (col, feat_name) in enumerate(zip(week0_cols, WEEK_FEATURES)):
    ax = axes[i]
    data = df_week[col].dropna()
    ax.hist(data, bins=50, color=PALETTE["primary"], alpha=0.7, edgecolor="white")
    median_val = data.median()
    ax.axvline(median_val, color=PALETTE["negative"], linestyle="--", linewidth=1,
               label=f"median={median_val:.2f}")
    ax.set_title(feat_name, fontweight="bold", fontsize=10)
    ax.legend(fontsize=7)

# Hide unused subplots
for j in range(n_feats, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Feature Distributions — Week 0 (Most Recent Week)", fontweight="bold", fontsize=14, y=1.01)
fig.tight_layout()

save_figure(fig, "02_week0_feature_distributions", subdir="eda", close=False)
plt.show()
plt.close(fig)

### Distribution notes

Key differences from the day approach:

- **Less zero-inflated**: Weekly aggregates almost always contain some training
  (even if individual days were rest days). `nr_rest_days` captures the rest pattern
  explicitly.

- **Min/max/avg triples**: The subjective metrics (`exertion`, `training_success`,
  `recovery`) now have three columns each — `avg`, `min`, `max`. This captures
  **within-week variation**: a week with one very hard session shows up as a gap
  between `avg_exertion` and `max_exertion`.

- **Count features**: `nr_sessions`, `nr_tough_sessions`, `nr_days_with_interval_session`
  are discrete integers, showing typical training patterns (e.g., most athletes
  do 1–2 interval sessions per week).

---

## 6. Day vs. Week Approach Comparison

The two approaches represent fundamentally different views of the **same underlying
training data**. Understanding their differences informs which approach might be
more effective for prediction.

In [ ]:
df_day = load_day_data()

day_positive_pct = df_day[INJURY_COL].mean() * 100
week_positive_pct = (binarize_target(df_week[INJURY_COL], threshold=WEEK_INJURY_THRESHOLD)).mean() * 100

comparison = pd.DataFrame({
    "Aspect": [
        "Rows",
        "Total columns",
        "Feature columns",
        "Temporal blocks",
        "Features per block",
        "Unique features",
        "Target type",
        "Positive class %",
        "Sentinel values",
    ],
    "Day Approach": [
        f"{df_day.shape[0]:,}",
        str(df_day.shape[1]),
        str(len(get_feature_columns(df_day))),
        "7 (days)",
        "10",
        "Raw daily metrics",
        "Binary (0/1)",
        f"{day_positive_pct:.2f}%",
        "Yes (−0.01)",
    ],
    "Week Approach": [
        f"{df_week.shape[0]:,}",
        str(df_week.shape[1]),
        f"{len(feature_cols_week)} ({N_WEEK_BLOCKS}×{len(WEEK_FEATURES)} + {len(WEEK_RELATIVE_FEATURES)} ratios)",
        "3 (weeks) + 3 ratios",
        "22",
        "Aggregated + ratios",
        f"Continuous (binarized at {WEEK_INJURY_THRESHOLD})",
        f"{week_positive_pct:.2f}%",
        "No",
    ],
})

print(comparison.to_string(index=False))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# Day approach: bar chart
day_counts = df_day[INJURY_COL].value_counts()
day_labels = ["No Injury (0)", "Injury (1)"]
ax1.barh(day_labels, [day_counts[0], day_counts[1]], color=[INJURY_PALETTE[0], INJURY_PALETTE[1]], height=0.5)
ax1.set_xlabel("Count")
ax1.set_title(f"Day Approach — Binary Target\n({day_positive_pct:.2f}% positive)", fontweight="bold")
for i, (count, label) in enumerate(zip([day_counts[0], day_counts[1]], day_labels)):
    ax1.text(count + max(day_counts) * 0.01, i, f"n={count:,}", va="center", fontweight="bold", fontsize=9)
ax1.set_xlim(0, max(day_counts) * 1.15)
sns.despine(ax=ax1, left=True)

# Week approach: histogram
ax2.hist(df_week[INJURY_COL], bins=80, color=PALETTE["primary"], alpha=0.7, edgecolor="white")
ax2.axvline(WEEK_INJURY_THRESHOLD, color=PALETTE["negative"], linestyle="--", linewidth=2,
            label=f"Threshold = {WEEK_INJURY_THRESHOLD} → {week_positive_pct:.2f}% positive")
ax2.set_xlabel("Injury score")
ax2.set_ylabel("Count")
ax2.set_title("Week Approach — Continuous Target", fontweight="bold")
ax2.legend(fontsize=9)
sns.despine(ax=ax2)

fig.suptitle("Target Distribution Comparison: Day vs. Week", fontweight="bold", fontsize=14, y=1.05)
fig.tight_layout()

save_figure(fig, "02_day_vs_week_targets", subdir="eda", close=False)
plt.show()
plt.close(fig)

### Interpretation

The two approaches represent a **resolution vs. richness trade-off**:

| | Day Approach | Week Approach |
|---|---|---|
| **Temporal resolution** | High (daily) | Lower (weekly) |
| **Feature richness** | Basic (10 raw metrics) | Rich (22 aggregated + ratios) |
| **Class imbalance** | Extreme (~1.36%) | Less extreme (depends on threshold) |
| **Sentinel handling** | Required | Not needed |
| **Unique advantage** | Daily patterns, day-to-day variation | Within-week variation, ACWR ratios |

**Counterintuitive finding from the paper**: Despite having fewer and simpler features,
the **day approach performed better** (AUC-ROC 0.724 vs. 0.678). This suggests that
**daily temporal granularity captures injury-relevant patterns** — like a sudden spike
on a single day — that weekly aggregation smooths out.

In sports science terms: an athlete might have a "normal" weekly total_km but one
extremely hard day within that week. The day approach captures this anomaly directly;
the week approach dilutes it into an average.

---

## 7. Key Insights and Implications

### Summary of findings

1. **Continuous target is heavily zero-inflated**: The vast majority of weeks show
   zero injury signal. Binarization at 0.5 produces a less extreme imbalance than
   the day approach, but the problem remains imbalanced.

2. **Threshold choice significantly affects class balance**: Moving from 0.5 to 0.25
   roughly doubles the positive class. The sensitivity analysis justifies the paper's
   choice of 0.5 while documenting alternatives for future experimentation (ADR-002).

3. **Ratio features encode acute:chronic workload changes**: The relative km ratios
   are the week approach's unique predictive asset, directly encoding training load
   spikes and drops — the strongest single predictor of injury in sports science
   literature.

4. **Richer features, coarser resolution**: 22 features per week (with min/max/avg
   triples) vs. 10 raw daily features. The min/max/avg encoding captures within-week
   variation but loses daily granularity.

5. **No sentinel values in week data**: Weekly aggregation naturally eliminates the
   rest-day encoding problem — rest days are captured by `nr_rest_days` and low values
   in other features, rather than sentinel values.

### Next steps

→ **Notebook 03**: Implement preprocessing for both approaches — sentinel replacement
(day), target binarization (week), scaling, and athlete-level train/test split.

→ **Notebook 04–05**: Model training and evaluation, where we'll see if the day
approach's AUC advantage holds in our replication.

In [ ]:
# Clean up temporary column
df_week.drop(columns=["injury_binary"], inplace=True)